In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from boardgames_recsys.data.filtering import filter_df
from boardgames_recsys.models.collaborative_filtering import calc_distance_matrix, get_KNN
from boardgames_recsys.data.matrix import get_matrix_user_game

In [ ]:
# import DB et set min_reviews
folder = "database_cleaned"
avis_clean  = pd.read_csv(f"{folder}/avis_clean.csv", index_col=0)
jeux_clean  = pd.read_csv(f"{folder}/jeux_clean.csv", index_col=0)
users       = pd.read_csv(f"trictrac_database/users.csv", names=["Username", "User id"])



# Global vision on users distances distribution [sample = 5 users]

In [ ]:
# Filter for min_reviews for users & games
min_reviews = 5
rev_filter = filter_df(avis_clean, min_reviews)

In [ ]:
matrix_ratings, mask_ratings, users_table, games_table = get_matrix_user_game(rev_filter)
# cos_sim_matrix = calc_distance_matrix(matrix_ratings, mask_ratings, "cos")
# cos_sim_matrix
cos_sim_matrix = np.load("generated_data/cos_sim_matrix_min5.npy")

Division of users by reviews
1. plus que 400
2. entre 100 et 400
3. de 6 à 100

In [ ]:
# Find users (5) for who we will plot distances. Create dataframe : User index (in matrix), User id (in DB), Count reviews (per user)

# Case 1 : top 5 most active
#users_ids = rev_filter[["User id", "Game id"]].groupby("User id", as_index=True).count().sort_values("Game id", ascending=False).head(5) 

# Case 2 : random sample for users in a category 
users_ids = rev_filter[["User id", "Game id"]].groupby("User id", as_index=True).count()
users_ids = users_ids[(users_ids["Game id"] >= 6) & (users_ids["Game id"] < 100)].sample(5)

assoc = users_table.to_frame().merge(users_ids, left_on="User id", right_index=True).reset_index()
assoc.columns = ["User index", "User id", "Count reviews"]
user_indices = assoc["User index"].to_numpy()

users_ids

In [ ]:
# Avoid taking distance to user himself (located at the matrix diagonal)
users_dist = np.array([np.delete(cos_sim_matrix[user], user) for user in user_indices]) 
users_dist.shape

In [ ]:
user_distances = np.round(users_dist, 2)   # eliminate distance precision 
dist_for_df  = user_distances.ravel()          # N-D to 1-D array
users_for_df = np.repeat(user_indices, 3002) # repeat each user's index k times (to construct dataframe in the next cell)
users_for_df.shape, dist_for_df.shape

In [ ]:
# Constuct Dataframe : where to each user index we associate an array of distances to ALL other users (except user himself)
# Dataframe is in the form (example for user index = 0)
# User index | Distance
# 0          | d1 (to user 1)
# 0          | d2 (to user 2)
# 0          | d3 (to user 3)

distance_to_users = pd.DataFrame({"Distance": dist_for_df, "User index":users_for_df}).sort_values("Distance")
distance_to_users = distance_to_users.groupby(["Distance", "User index"], as_index=True).size().reset_index(name="Number of users")
distance_to_users

In [ ]:
sns.set_theme(rc={'figure.figsize':(14,6)})

ax = sns.pointplot(distance_to_users, x="Distance", y="Number of users", hue="User index", scale=0.5,  palette="Spectral")

for ind, label in enumerate(ax.get_xticklabels()):
    if ind % 3 == 0:  # every 10th label is kept
        label.set_visible(True)
    else:
        label.set_visible(False)
ax.figure.suptitle("5 users [6 - 100 reviews]")
# g = sns.FacetGrid(distance_to_users, col="User index", col_wrap=4, xlim=(0.4, 1.))
# g.map(sns.scatterplot, "Distance", "Number of users")
# g.figure.subplots_adjust(top=0.9) 
# g.figure.suptitle("Users [100-420 reviews]. k = 54 cos")